In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np


import pybedtools
from pybedtools import BedTool


#For plotting
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

#For statistics
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools

import re
from Bio import SeqIO
import ast # for safe eveal, for parsing some of the data
import math
# --- Shared helpers: const.py lives in analyses/common ---
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'common')))

#import importlib
#importlib.reload(const)

import const #to reload use import(importlib) and then importlib.reload(const)
from const import pos_active_ctrl_color,neg_active_ctrl_color,highlight_color,custom_cmap
from const import set_equal_plot_limits
from const import plot_color_pallete
from const import custom_cmap_bolder
from const import FONT_SIZE_small
const.set_plot_style()
import matplotlib.ticker as mtick

# --- Working directory (EDIT): used for relative .bed/.csv I/O in some notebooks ---
WORK_DIR = '/home/labs/davidgo/Collaboration/backup/humanMPRA/scripts/produce_paper_figures'
os.chdir(WORK_DIR)

output_path = '/home/labs/davidgo/Collaboration/humanMPRA/paper/raw_plots/'

# Parameters

In [ ]:
# Use CPM normalization?
cpm=True

# Use log scale in visualization?
logScale=False

#Additional filters?
min_DNA_reads = 5 

min_DNA_counts = 0

# Figure 1 - hMPRA validation

### import and process all libraries

In [ ]:
full_activity_df = pd.read_csv('/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/quantitative_analysis_combined/comb_df_combined_fdr.csv')


#### Apply filters

In [ ]:
# Keep only rows where the 'oligo' column contains "SCREEN"
print(f"oligos before filtering for SCREEN overlap:{len(full_activity_df)}")
full_activity_df = full_activity_df[full_activity_df["oligo"].str.contains("SCREEN", na=False)]
print(f"oligos after filtering for SCREEN overlap:{len(full_activity_df)}")
full_activity_df =  full_activity_df[~full_activity_df["oligo"].str.contains("Ctrl", na=False)]
print(f"oligos after filtering for non-controls:{len(full_activity_df)}")

min_DNA_counts
if min_DNA_counts>0:
    full_activity_df =  full_activity_df[(full_activity_df['DNA_rep_comb']>min_DNA_counts)]
    print(f"oligos after filtering for oligos with over 10 DNA counts (combined for all replicates):{len(full_activity_df)}")



# remove oligos which were later removed
full_activity_df = full_activity_df[~(full_activity_df['orientation_fix']=='fixed_in_L4')&
                           ~(full_activity_df['oligo'].str.contains('SCREEN_EH'))&
                          ~(full_activity_df['oligo'].str.contains('hh.missing.oligos')) ]
print(f"oligos after filtering out oligos which were later remoed due to L4a1 (eg orientation fix):{len(full_activity_df)}")

#if min_barcodes>0:
#    full_activity_df =  full_activity_df[(full_activity_df['bcs_DNA_rep1']>10) &(full_activity_df['bcs_DNA_rep2']>10)]
#    print(f"oligos after filtering for oligos with over 10 barcodesin both reps 1 and 2:{len(full_activity_df)}")


In [ ]:
# Add normalization of RNA and DNA counts
DNA_sum = full_activity_df["DNA_rep_comb"].sum()
RNA_sum = full_activity_df["RNA_rep_comb"].sum()

full_activity_df["DNA_rep_comb_cpm"] = 1000000*(full_activity_df["DNA_rep_comb"]+1)/DNA_sum
full_activity_df["RNA_rep_comb_cpm"] = 1000000*(full_activity_df["RNA_rep_comb"]+1)/RNA_sum

full_activity_df["DNA_rep_comb_cpm_log"] = np.log2(full_activity_df["DNA_rep_comb_cpm"])
full_activity_df["RNA_rep_comb_cpm_log"] = np.log2(full_activity_df["RNA_rep_comb_cpm"])


In [ ]:
full_comparative_df = pd.read_csv("/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/comparative_analysis_combined/mpranalyze_comp_res_fdr_complete.csv")
full_comparative_df = full_comparative_df[['oligo','logFC','differntial.wo_controls']]
full_comparative_df = full_comparative_df.rename(
    columns={'differntial.wo_controls': 'differential_activity'}
)


## Drop shape

In [ ]:
# Prepare the data
# Replace Inf values with NaN, then drop any rows with NaN values
replicates_activity_df = full_activity_df[['oligo','ratio_log_rep1','ratio_log_rep2','bcs_DNA_rep1','bcs_DNA_rep2']].copy()

print(f'Number of oligos: {len(replicates_activity_df)}')

replicates_activity_df.replace([np.inf, -np.inf], np.nan)

# Drop rows where either 'ratio_log_filtered_std2_rep1' or 'ratio_log_filtered_std2_rep2' has NaN or Inf
replicates_activity_df = replicates_activity_df.dropna(subset=['ratio_log_rep1', 'ratio_log_rep2'])

#replicates_activity_df = replicates_activity_df.sample(n=60000, random_state=42)

x = replicates_activity_df['ratio_log_rep1'].values
y = replicates_activity_df['ratio_log_rep2'].values


In [ ]:
plt.clf()

fig, ax = plt.subplots()

hb = ax.hexbin(
    x, y,
    gridsize=200,          # adjust for coarser/finer grid
    cmap=custom_cmap_bolder,
    mincnt=1,             # only show hexes with at least 1 point
    bins='log'            # color ~ log(count); remove if you want raw counts
    # alpha=0.9           # optional, usually you can skip alpha for hexbin
)

# Add colorbar
cb = fig.colorbar(hb, ax=ax)
cb.set_label(r'Density ($\log_{10}$)')

ax.set_xlabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 1')
ax.set_ylabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$ replicate 2')

# Keep your tick simplification
xticks = ax.get_xticks()
yticks = ax.get_yticks()
ax.set_xticks([xticks[0], xticks[-1]])
ax.set_yticks([yticks[0], yticks[-1]])

ax.set_aspect('equal', adjustable='box')
# Optional: set explicit limits if you want
# ax.set_xlim(-4, 6)
# ax.set_ylim(-4, 6)

const.save_fig(plt, 'RNA_DNA_ratio_correlation_between_replicates', output_path)
plt.show()


## Activity histogram

In [ ]:
bin_edges = np.linspace(-4, 4, 201)  # 100 bins between -10 and 20

# First histogram (all scores)
plt.hist(
    data=full_activity_df,
    x='ratio_log_rep_comb', bins=bin_edges, color=plot_color_pallete['default_color'], label='Non-active cCREs')

# Second histogram (filtered scores)
plt.hist(
    data=full_activity_df[full_activity_df['activity_adjusted'] == 'active'],
    x='ratio_log_rep_comb', bins=bin_edges, color='red', label='Active cCREs')

plt.xlabel(r'$\log_{2}\!\left(\frac{\mathrm{RNA}}{\mathrm{DNA}}\right)$')
plt.ylabel('Number of cCREs')
plt.legend()
stat,pval=stats.skewtest(full_activity_df['ratio_log_rep_comb'].dropna())

# Smallest positive float in Python
min_p = np.nextafter(0, 1)  # ~5e-324 for float64

if pval < min_p:
    p_text = f"p < {min_p:.1e}"
else:
    p_text = f"p = {pval:.3g}"

plt.text(
    0.05, 0.95,  # relative position in axes coordinates
    f"Skew test:\nstat = {stat:.2f}\n{p_text}",
    ha='left', va='top',
    transform=plt.gca().transAxes,
    fontsize=10,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
)


plt.xticks([-4,0,4])

yticks = plt.gca().get_yticks()
plt.yticks([yticks[0], yticks[-1]])

plt.legend(
    loc="upper right",
    fontsize=10,         # smaller text
    markerscale=0.8,    # shrink the color boxes
    handlelength=1,     # shorter legend lines
    handletextpad=0.5,  # less padding between box and text
    borderpad=0.3       # tighter box
)


const.save_fig(plt,'activity_distribution',output_path)
plt.show()


##  Controls boxplot


In [ ]:
# List of libraries
libraries = [
    "L1a1", "L1a2", "L1a3",
    "L2a1", "L2a2", "L2a3",
    "L3a1", "L3a2", "L3a3",
    "L4a1"
]

base_dir = r"/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes"

dfs = []

for lib in libraries:
    csv_path = os.path.join(
        base_dir,
        lib,
        "output",
        "activity_after_filter",
        "comb_df_adjusted_fdr.csv"
    )

    # Read CSV
    df = pd.read_csv(csv_path)

    # Keep only rows where "oligo" contains "ctrl"
    df_ctrl = df[
    df["oligo"].str.contains("Ctrl", na=False, case=False) |
    df["oligo"].str.contains("scramble", na=False, case=False)
        ].copy()
    
    # Optional: add a column to remember which library it came from
    df_ctrl["library"] = lib

    dfs.append(df_ctrl)

# Combine all control oligos from all libraries into one df
all_ctrl_oligos = pd.concat(dfs, ignore_index=True)


In [ ]:
control_types = [
    "NegCtrl_active_not_diff_ESCs+Osteoblasts+NPCs",
    "NegCtrl_non_SCREEN",
    "scrambled_control",
    "PosCtrl_diff_ESCs_Weiss_seq11687_derived_a1_L4",
    "NegCtrl_not_active_ESCs+Osteoblasts+NPCs",
    "PosCtrl_diff_ESCs+Osteoblasts+NPCs",
    "PosCtrl_chondrocyte_active",
    "PosCtrl_diff_NPCs_Weiss",
    "PosCtrl_diff_ESCs+NPCs",
    "PosCtrl_diff_ESCs_Weiss",
    "PosCtrl_diff_ESCs+Osteoblasts",
    "PosCtrl_diff_Osteoblasts+NPCs",
    "PosCtrl_diff_Osteoblasts",
    "PosCtrl_neuron_active",
    "PosCtrl_osteoblast_active",
]

ctrl_dict = {}

for ctype in control_types:
    mask = all_ctrl_oligos["oligo"].str.contains(ctype, regex=False)
    ctrl_dict[ctype] = all_ctrl_oligos[mask].copy()

# Optional: if you want a df of any remaining controls that didn't match:
unmatched_mask = ~pd.concat(
    [all_ctrl_oligos["oligo"].str.contains(ct, na=False, regex=False) for ct in control_types],
    axis=1
).any(axis=1)

ctrl_dict["other_ctrls"] = all_ctrl_oligos[unmatched_mask].copy()

to_combine = [
    "PosCtrl_chondrocyte_active",
    "PosCtrl_osteoblast_active",
    "NegCtrl_active_not_diff_ESCs+Osteoblasts+NPCs"
]

ctrl_dict["all_activity_pos_ctrl"] = (
    pd.concat([ctrl_dict[k] for k in to_combine], ignore_index=True)
)

#Add the tested cCREs
# Add tested cCREs as a new entry in the control dictionary
# (here we only keep mad.score since that's what we plot)
ctrl_dict["tested_cCREs"] = full_activity_df[["mad.score"]]



In [ ]:
pos_color  = '#8FBCE1'
neg_color  = '#DE2326'
high_color = '#BBA9D2'

# Keys from the dictionary and their pretty labels
plot_keys = {
    "scrambled_control": "Negative control (scrambled)",
    "NegCtrl_non_SCREEN": "Negative control (non-SCREEN)",
    "all_activity_pos_ctrl": "Positive control (active)",
    "NegCtrl_not_active_ESCs+Osteoblasts+NPCs": "Negative control (inactive)",
    "tested_cCREs": "cCREs",
}

plot_dfs = []
for key, pretty in plot_keys.items():
    if key not in ctrl_dict:
        continue
    df_temp = ctrl_dict[key].copy()
    df_temp["control_type"] = key
    df_temp["control_type_pretty"] = pretty
    plot_dfs.append(df_temp)

c = pd.concat(plot_dfs, ignore_index=True)

# Order of boxplots (y-axis)
order = [
    "Positive control (active)",
    "Negative control (inactive)",
    "Negative control (scrambled)",
    "Negative control (non-SCREEN)",
    "cCREs",
]

# Matching colors
palette = [
    pos_color,  # Positive control (active)
    neg_color,  # Negative control (inactive)
    neg_color,  # Negative control (scrambled)
    neg_color,  # Negative control (non-SCREEN)
    high_color, # cCREs
]
sns.boxplot(
    data=c,
    x="mad.score",
    y="control_type_pretty",
    showfliers=True,
    linewidth=2,
    order=order,
    palette=palette
)
plt.xlim(-10,30)

plt.ylabel("")
plt.xlabel("Activity statistic")
plt.show()

In [ ]:
c_capped = c[c['mad.score']<30]
sns.violinplot(
    data=c_capped,
    x="mad.score",
    y="control_type_pretty",
    #showfliers=False,
    linewidth=2,
    order=order,
    palette=palette
)
plt.ylabel("")
plt.xlabel("Activity statistic")

## RNA vs DNA

In [ ]:
# Prepare the data
x = full_activity_df['DNA_rep_comb'].values
y = full_activity_df['RNA_rep_comb'].values

# Build a mask removing NaN and ±inf in both x and y
mask = (
    np.isfinite(x) &
    np.isfinite(y) &
    (y < 300_000)
)

# Apply mask
x = x[mask]
y = y[mask]

print(len(x), "points remain after filtering.")


# Evaluate the KDE on the grid
#values = np.vstack([x, y])
#kernel = gaussian_kde(values)
#density = kernel(values)

In [ ]:
plt.clf()
fig, ax = plt.subplots()

# Hexbin density plot instead of scatter
hb = ax.hexbin(
    x, y,
    gridsize=200,              # tweak for smoother/coarser bins
    cmap=custom_cmap_bolder,  # same colormap as before
    mincnt=1,                 # only show bins with at least 1 point
    bins='log'                # log10 of counts
)

# Y = X diagonal
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()

diag_min = min(xmin, ymin)
diag_max = max(xmax, ymax)

ax.plot(
    [diag_min, diag_max],
    [diag_min, diag_max],
    color='black',
    linestyle='--',
    linewidth=1.2,
    alpha=0.8
)

# Colorbar with label
cb = fig.colorbar(hb, ax=ax)
cb.set_label(r'Density ($\log_{10}$)')

# Set axis limits
ax.set_xlim(0, max(x))
ax.set_ylim(0, max(y))

# Simplify ticks (first & last)
xticks = ax.get_xticks()
yticks = ax.get_yticks()
ax.set_xticks([xticks[0], xticks[-1]])
ax.set_yticks([yticks[0], yticks[-1]])

# Axis labels
ax.set_xlabel('DNA count')
ax.set_ylabel('RNA count')

const.save_fig(plt, 'RNA_DNA_ratio', output_path)
plt.show()


## ancestral vs derived alleles

In [ ]:
ancestral_derived_data = full_activity_df.copy()
print(len(ancestral_derived_data))
ancestral_derived_data = ancestral_derived_data[ancestral_derived_data['count_rep_comb']>10]
#ancestral_derived_data = ancestral_derived_data[ancestral_derived_data['activity_adjusted_combined']=='active']
print(len(ancestral_derived_data))

# 1. Keep only rows that are ancestral/derived
ancestral_derived_data = ancestral_derived_data[ancestral_derived_data["oligo"].str.contains("ancestral|derived", case=False, na=False)].copy()


# 2. Add a column indicating whether it's ancestral or derived
ancestral_derived_data["status"] = np.where(
    ancestral_derived_data["oligo"].str.contains("ancestral", case=False, na=False),
    "ancestral",
    "derived"
)

# 3. Define a "pair_id" by removing the ancestral/derived part from the oligo name
def make_pair_id(s):
    # remove '_ancestral', '-ancestral', ' ancestral', etc. (same for derived)
    s = re.sub(r"[_\- ]?(ancestral|derived)", "", s, flags=re.IGNORECASE)
    return s

ancestral_derived_data["pair_id"] = ancestral_derived_data["oligo"].apply(make_pair_id)


# 4. Pivot so each row = one pair, with ancestral / derived mad.score as columns
ancestral_derived_data = (
    ancestral_derived_data
    .pivot_table(
        index="pair_id",
        columns="status",
        values="mad.score",
        aggfunc="first"   # in case there are duplicates
    )
    .reset_index()
)

#4. Merge with differential activity df 
ancestral_derived_data = ancestral_derived_data.merge(
    full_comparative_df,
    how='left',
    left_on='pair_id',
    right_on='oligo'
)

# Columns are now: ['pair_id', 'ancestral', 'derived']
# 5. Filter to keep only pairs where both values are real numbers (no NaN/inf)
mask_valid = (
    ancestral_derived_data["ancestral"].notna() &
    ancestral_derived_data["derived"].notna() &
    np.isfinite(ancestral_derived_data["ancestral"]) &
    np.isfinite(ancestral_derived_data["derived"])
)

ancestral_derived_data = ancestral_derived_data.dropna()

# Remove +inf or -inf
ancestral_derived_data = ancestral_derived_data[~ancestral_derived_data.isin([np.inf, -np.inf]).any(axis=1)]
#ancestral_derived_data = ancestral_derived_data.sample(n=100000, random_state=42)

In [ ]:
x = ancestral_derived_data['ancestral'].values
y = ancestral_derived_data['derived'].values
colors = ancestral_derived_data['differential_activity'].values

# Create the KDE (Kernel Density Estimate)
#values = np.vstack([x, y])
#kernel = gaussian_kde(values)

# Evaluate the KDE for each data point
#density = kernel(values)

#max_density_threshold = 0.1 

# Density maximum value clipping 
#density_capped = np.clip(density, a_min=None, a_max=max_density_threshold)

In [ ]:
plt.clf()

scatter = plt.scatter(
    x, y,
    #c=colors,               # color by differential.wo_controls
    alpha=0.1,
    s=10,
    edgecolors='none'
)

scatter = plt.scatter(
    x[colors==True], y[colors==True],
    c='red',               # color by differential.wo_controls
    s=10,
    edgecolors='none'
)


#plt.colorbar(scatter, label="Differential (wo controls)")

plt.xlabel(r'Ancestral allele mad score')
plt.ylabel(r'Derived allele mad score')

plt.xlim(0, 100)
plt.ylim(0,100)
xticks = plt.xticks()[0]
yticks = plt.yticks()[0]
plt.xticks([xticks[0], xticks[-1]])
plt.yticks([yticks[0], yticks[-1]])

const.save_fig(plt, 'ancestral_vs_derived_activity_scatter', output_path)
plt.show()
